# Lab: Deploying DeepSeek-R1 on AMD MI300X

In this lab, you'll deploy the DeepSeek-R1-Distill-Llama-70B reasoning model using vLLM on AMD Instinct MI300X hardware with optimized settings, then test its inference capabilities.

**Model:** `deepseek-ai/DeepSeek-R1-Distill-Llama-70B`  
**Hardware:** AMD Instinct MI300X (192GB VRAM)  
**Framework:** vLLM with ROCm and AITER optimizations

## 1. Check GPU Memory

Before loading a 70B parameter model, let's verify how much VRAM is available on our MI300X.

**Memory requirement:** ~140GB (70B parameters × 2 bytes for BF16)

In [ ]:
!rocm-smi --showmeminfo vram

## 2. Configure AITER Environment Variables

Set up ROCm and AITER optimizations for best performance on MI300X.

### Distributed Communication Variables

| Variable | Purpose |
|----------|---------|
| `GLOO_SOCKET_IFNAME=eth0` | PyTorch distributed backend for process communication |
| `NCCL_SOCKET_IFNAME=eth0` | RCCL (AMD's NCCL equivalent) for heavy data movement between processes |

### vLLM ROCm/AITER Variables

| Variable | Purpose |
|----------|---------|
| `VLLM_ROCM_USE_AITER=1` | Master switch — uses AITER's optimized C++ kernels |
| `VLLM_ROCM_USE_AITER_TRITON_FUSED_ROPE_ZEROS_KV_CACHE=1` | Fuses KV cache dequantization and RoPE calculation into one step |
| `VLLM_ROCM_USE_AITER_LINEAR=1` | Routes matrix multiplications through optimized AMD kernels |
| `TORCH_BLAS_PREFER_HIPBLASLT=1` | Prefer hipBLASLt for matrix ops |

In [ ]:
import os
os.environ["VLLM_ROCM_USE_AITER"] = "1"
os.environ["VLLM_ROCM_USE_AITER_TRITON_FUSED_ROPE_ZEROS_KV_CACHE"] = "1"
os.environ["VLLM_ROCM_USE_AITER_LINEAR"] = "1"
os.environ["TORCH_BLAS_PREFER_HIPBLASLT"] = "1"
os.environ["GLOO_SOCKET_IFNAME"] = "eth0"
os.environ["NCCL_SOCKET_IFNAME"] = "eth0"

print("✅ Environment variables set")

## 3. Launch DeepSeek-R1 with Optimized Settings

Now we'll create and run a launch script that starts vLLM with settings optimized for the MI300X.

### vLLM Server Parameters

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `--tensor-parallel-size` | 1 | Single GPU deployment |
| `--max-model-len` | 8192 | Context window (larger = more VRAM for KV cache) |
| `--dtype` | bfloat16 | Native 16-bit format for MI300X |
| `--kv-cache-dtype` | fp8 | Halves KV cache memory, doubling capacity for users/context |
| `--gpu-memory-utilization` | 0.92 | Leaves ~15GB for HIP graphs, fragmentation, kernel workspace |
| `--no-enable-chunked-prefill` | — | Disables chunking — maximizes throughput at cost of latency spikes |
| `--max-num-seqs` | 512 | Double default — keeps MI300X compute units busy |

In [ ]:
%%bash --bg --out vllm_log --err vllm_err
python -m vllm.entrypoints.openai.api_server \
    --model deepseek-ai/DeepSeek-R1-Distill-Llama-70B \
    --tensor-parallel-size 1 \
    --max-model-len 8192 \
    --trust-remote-code \
    --dtype bfloat16 \
    --kv-cache-dtype fp8 \
    --gpu-memory-utilization 0.92 \
    --no-enable-chunked-prefill \
    --max-num-seqs 512 \
    --port 8080

## 4. Wait for Model Loading

The 70B model needs to be downloaded from HuggingFace (if not cached) and loaded into GPU memory. This can take several minutes.

In [ ]:
import requests
import time

print("Waiting for DeepSeek model to load...")
print("(70B model download + initialization can take several minutes)\n")

for i in range(60):
    try:
        r = requests.get("http://localhost:8080/v1/models", timeout=5)
        if r.status_code == 200:
            print("\n✅ DeepSeek server ready!")
            models = r.json()
            print(f"Model: {models['data'][0]['id']}")
            break
    except:
        pass
    print(f"Loading... ({i+1}/60) - {(i+1)*10}s elapsed")
    time.sleep(10)
else:
    print("⚠️ Server didn't start in time.")

## 5. Test Reasoning Capabilities

DeepSeek-R1 is a **reasoning model** that shows its thinking process. Let's test it with a math problem to see step-by-step reasoning in action.

**Note:** DeepSeek-R1 outputs its internal reasoning wrapped in `<think>...</think>` tags, followed by the final answer. This is the model's "chain of thought" visible to users.

In [ ]:
import requests

response = requests.post(
    "http://localhost:8080/v1/chat/completions",
    json={
        "model": "deepseek-ai/DeepSeek-R1-Distill-Llama-70B",
        "messages": [
            {"role": "user", "content": "What is 25 * 37? Think step by step."}
        ],
        "max_tokens": 512
    }
)

result = response.json()
print("DeepSeek-R1 Response:\n")
print(result["choices"][0]["message"]["content"])

### Classic Reasoning Puzzle

The "bat and ball" problem is a famous cognitive reflection test. The intuitive (wrong) answer is 0.10, but proper algebraic reasoning gives 0.05.

A true reasoning model should work through the algebra correctly rather than jumping to the intuitive answer.

In [ ]:
response = requests.post(
    "http://localhost:8080/v1/chat/completions",
    json={
        "model": "deepseek-ai/DeepSeek-R1-Distill-Llama-70B",
        "messages": [
            {"role": "user", "content": "A bat and ball cost $1.10 total. The bat costs $1.00 more than the ball. How much does the ball cost?"}
        ],
        "max_tokens": 1024
    }
)

print("Classic reasoning puzzle:\n")
print(response.json()["choices"][0]["message"]["content"])

## 6. Run Benchmark

The benchmark suite is pre-installed. It sends **800 requests** with **50 concurrent connections** to measure inference throughput.

### Understanding the Memory Wall

**Expected maximum:** ~24 tokens/second for single-user decode

Why? This is limited by memory bandwidth, not compute:

| Factor | Value |
|--------|-------|
| MI300X bandwidth | ~5.3 TB/s theoretical |
| Real-world efficiency | ~60-65% → **3.3 TB/s** |
| Model size (70B × BF16) | **140 GB** |
| Max tokens/sec | 3.3 TB/s ÷ 140 GB ≈ **24 tok/s** |

The GPU must read all 140GB of weights for *each token* generated. This is the "memory wall."

> **Note:** The optimizations we configured (AITER, FP8 KV cache, disabled chunked prefill) shine under **high concurrency** and **long context** — not single-user benchmarks.

**Benchmarking Scripts**: The scripts used in this lab are open source and available at [github.com/AMD-AGI/GPU-Optimization-for-LLM-Inference](https://github.com/AMD-AGI/GPU-Optimization-for-LLM-Inference).

In [ ]:
!python /workspace/GPU-Optimization-for-LLM-Inference/summary_benchmark.py --port 8080 --model "deepseek-ai/DeepSeek-R1-Distill-Llama-70B"